In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import paths   # input/output locations: set SN2024VJM_DATA / SN2024VJM_EXTERNAL (see README)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib widget

In [ ]:
spec = pd.read_csv(paths.SPECTRA / 'AT2024vjm_Xshooter_20240916_UBV_VIS.txt',sep=' ')
z = 0.0028
# Function to calculate equivalent width
def EW(wavelength, flux, line_center, window=2):
    """
    Calculate the equivalent width of an absorption line.
    :param wavelength: Wavelength array
    :param flux: Normalized flux array
    :param line_center: Central wavelength of the absorption line
    :param window: Range around the line center to integrate (in Angstroms)
    :return: Equivalent width in Angstroms
    """
    # Select wavelength range around the absorption line
    mask = (wavelength > (line_center - window)) & (wavelength < (line_center + window))
    wl_line = wavelength[mask]
    flux_line = flux[mask]
    
    # Compute EW using numerical integration (trapezoidal rule)
    EW = np.trapz(1 - flux_line, wl_line)
    plt.fill_between(wl_line / (1 + z), flux_line, 1, color='r' if line_center == D1 else 'b', alpha=0.3)
    return EW

In [ ]:
# Functions to fit and remove polynomial background
def fit_poly(wl, flux, n, omissions=[]):
    """
    Fit a polynomial while omitting given wavelength regions.
    """
    if omissions:
        mask = np.ones_like(wl, dtype=bool)
        for omission in omissions:
            mask &= (wl < omission[0]) | (wl > omission[1])
        wl_temp = wl[mask]
        flux_temp = flux[mask]
    else:
        wl_temp = wl
        flux_temp = flux
    
    pfit = np.polyfit(wl_temp, flux_temp, n)
    return np.polyval(pfit, wl)

def remove_poly(wl, flux, n=5, omissions=[]):
    """
    Remove polynomial background while excluding specific wavelength regions.
    """
    y = fit_poly(wl, flux, n, omissions)
    return flux / y

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = 0.003
# Define the rest-frame wavelengths of the Na I D lines
D1_rest = 5896  # Angstroms
D2_rest = 5890  # Angstroms

# Compute observed wavelengths given redshift z
D1 = D1_rest * (1 + z)
D2 = D2_rest * (1 + z)

# Define wavelength range for plotting and selection
lines_range = [D1 - 10, D2 + 10]
range = [D1 - 20, D2 + 20]

# Select data in the given range
cond = (spec['wavelength'] > range[0]) & (spec['wavelength'] < range[1])
wl = spec['wavelength'][cond]
flux = spec['flux'][cond]
flux = remove_poly(wl, flux, 3, omissions=[(D1-2, D1+2), (D2-2, D2+2)])

# Plot the spectrum
plt.figure()
plt.plot(wl / (1 + z), flux, 'k-', label='Spectrum')

# Compute equivalent widths
EW_D1 = EW(wl, flux, D1)
EW_D2 = EW(wl, flux, D2)

# Plot the Na lines
plt.vlines(D1_rest, 0, np.max(flux) + 0.1 * np.max(flux), colors='r', linestyles='dashed', label='Na D1')
plt.vlines(D2_rest, 0, np.max(flux) + 0.1 * np.max(flux), colors='b', linestyles='dashed', label='Na D2')
plt.xlabel('Wavelength (Å)')
plt.ylabel('Flux')
plt.legend()
plt.show()

print(f"Equivalent Width of Na I D1 (5896 Å): {EW_D1:.3f} Å")
print(f"Equivalent Width of Na I D2 (5890 Å): {EW_D2:.3f} Å")

# Compute extinction using Poznanski et al. 2012 formula
# E(B-V) = 0.78 * EW_NaI - 0.1 (Equation from Poznanski et al. 2012)
EW_NaI = EW_D1 + EW_D2  # Total Na I D equivalent width
E_BV = 10**(1.17 * EW_NaI - 1.85)
E_BV_minus =  E_BV - 10**(1.17 * EW_NaI - 1.85 -0.08)
E_BV_plus = 10**(1.17 * EW_NaI - 1.85 + 0.08) - E_BV

print("E(B-V) = {:.3f} + {:.3f} - {:.3f}".format(E_BV, E_BV_plus, E_BV_minus))
